# Ramping up with myogen on NinaPro DB5

myogen turns hand and wrist **joint angles** into the forearm **sEMG** an armband would record: movement → musculoskeletal forearm model → muscle activity → EMG at the electrodes → recording device.

NinaPro DB5 recorded exactly that pair: a CyberGlove (the movement) and two Myo armbands (the EMG). So we can feed the glove motion to myogen and compare the simulated EMG with the real one.

1. The data: what a DB5 recording contains
2. From glove to a `HandMotion` (a loader function)
3. Setting up a simulator: electrodes, subject
4. Running a simulation and reading the `Recording`
5. Several motions at once (parallel batch)
6. Real vs simulated EMG: signals and envelopes
7. Calibrating the device to the real recording
8. Saving, loading, and `mg.replay` in 3-D

Before running: `./setup.sh` and `./download_ninapro.sh` (see the README).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import myogen as mg
from ninapro import load_db5, load_mat

DB5 = Path("data/ninapro_db5/s1")
RECORDING = DB5 / "S1_E1_A1.mat"          # subject 1, exercise 1 (finger movements)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. The data

Each DB5 file is one exercise of one subject, sampled at 200 Hz:

- `emg` (T, 16): the two Myo armbands (8 channels each), in the Myo's raw int8 counts
- `glove` (T, 22): CyberGlove sensors (uncalibrated)
- `restimulus` (T,): which movement the subject was doing (0 = rest), `rerepetition` (T,): which repetition
- `acc` (T, 3): the Myo accelerometer (used for forearm rotation)

In [ ]:
rec = load_mat(RECORDING)
fs = rec["fs"]
print({k: v.shape for k, v in rec.items() if isinstance(v, np.ndarray)}, f"fs = {fs} Hz")
print(f"duration {len(rec['emg']) / fs / 60:.1f} min, movements {np.unique(rec['restimulus'])[1:]}")

# A minute of the recording, starting just before the first movement.
start = max(0, int(np.argmax(rec["restimulus"] > 0) - 5 * fs))
window = slice(start, start + int(60 * fs))
t = np.arange(window.stop - window.start) / fs

fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 2, 1]})
emg = rec["emg"][window]
for c in range(16):
    ax[0].plot(t, emg[:, c] / 128 + c, lw=0.4, color=f"C{c // 8}")
ax[0].set(yticks=range(16), yticklabels=[f"ch{c}" for c in range(16)], title="real EMG (Myo, 2 armbands)")
ax[1].plot(t, rec["glove"][window][:, [1, 5, 8, 12, 16]], lw=1)
ax[1].set(ylabel="glove sensors", title="CyberGlove (a few sensors)")
ax[2].plot(t, rec["restimulus"][window], color="k", lw=1)
ax[2].set(ylabel="movement", xlabel="time (s)")
plt.tight_layout()

## 2. From glove to a `HandMotion`

A `HandMotion` holds the model's **23 hand and wrist joint angles** (radians) over time, plus optional rest labels. myogen reads any format through a **loader**: a function `path -> motion` that returns the angles already in that format (or a ready `HandMotion`).

`ninapro.load_db5` (in this folder) is such a loader: it turns the glove sensors into joint angles (normalised over all of the subject's recordings) and hands them to `mg.JointAdapter`, which maps them onto the model's joints and fills any joint that was not measured.

In [ ]:
motion = mg.HandMotion.load(RECORDING, load_db5)
print(motion.angles.shape, motion.fs, "Hz,", f"{motion.duration / 60:.1f} min, rest {motion.rest.mean():.0%} of the time")

from myogen.simulation.musculoskeletal import HAND_JOINTS
print(HAND_JOINTS)


def segment(m: mg.HandMotion, window: slice) -> mg.HandMotion:
    """A piece of a motion (the full recordings are 10-20 min: a minute is plenty to play with)."""
    return mg.HandMotion(m.angles[window], m.fs, rest=m.rest[window])


minute = segment(motion, window)
shown = ["flexion_r", "pro_sup_r", "mcp2_flexion_r", "pm3_flexion_r", "cmc_abduction_r"]
plt.figure(figsize=(11, 3))
for j in shown:
    plt.plot(t, np.degrees(minute.angles[:, HAND_JOINTS.index(j)]), label=j)
plt.fill_between(t, -90, 90, where=minute.rest, color="0.9", label="rest")
plt.ylim(-60, 100)
plt.legend(ncol=6, fontsize=8, loc="upper right")
plt.gca().set(xlabel="time (s)", ylabel="angle (deg)", title="the minute we simulate, as model joint angles")
plt.tight_layout()

## 3. Setting up a simulator

A `Simulator` combines three things:

- **`Electrodes`**: where the electrodes sit and which device records them. `Electrodes.myo()` is the DB5 setup: two Myo rings, 16 channels, int8 counts at 200 Hz. Other presets: `ring()`, `otto_bock()`; or load your own YAML.
- **`Subject`**: the person (forearm size, fat, EMG gain, co-contraction habits, ...). `seed` sets its randomness: same seed, same EMG.
- **`SimParams`**: the model parameters (rarely needed).

In [ ]:
electrodes = mg.Electrodes.myo()
print(electrodes.device, electrodes.fs_out, "Hz,", len(electrodes), "channels:", electrodes.names[:4], "...")

subject = mg.Subject(seed=1)
print(f"forearm radius {subject.forearm_radius_mm} mm, fat {subject.fat_mm} mm, EMG gain {subject.gain_mv} mV")

sim = mg.Simulator(electrodes, subject=subject)

## 4. Running a simulation

`run` goes movement → muscles → EMG → device and shows its progress (the muscle controller first, then the EMG). It returns a `Recording`: the device's EMG plus labels at 100 Hz of what the hand and muscles did, and a quality check (`qc`) of how well the simulated hand followed the motion.

In [ ]:
recording = sim.run(minute)

print(recording.emg.shape, recording.fs, "Hz")
print("QC pass:", recording.qc["pass"], f"| tracking RMSE {recording.qc['rmse_deg']:.2f} deg")
print("labels:", list(recording.labels))

In [ ]:
lab, t_lab = recording.labels, np.arange(len(recording.labels["q"])) / recording.fs_labels
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
for j in ["flexion_r", "mcp2_flexion_r"]:
    k = recording.joint_names.index(j)
    ax[0].plot(t_lab, np.degrees(lab["q_ref"][:, k]), "k--", lw=1)
    ax[0].plot(t_lab, np.degrees(lab["q"][:, k]), lw=1.2, label=j)
ax[0].set(ylabel="deg", title="joint angles: wanted (dashed) and what the simulated hand did")
ax[0].legend(fontsize=8)
im = ax[1].imshow(lab["excitation"].T, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=np.percentile(lab["excitation"], 99.5),
                  extent=[0, t_lab[-1], 0, len(recording.muscle_names)])
ax[1].set_yticks(np.arange(len(recording.muscle_names))[::3] + 0.5, recording.muscle_names[::3], fontsize=6)
ax[1].set(xlabel="time (s)", title="muscle excitation (what drives the EMG)")
plt.colorbar(im, ax=ax[1], pad=0.01)
plt.tight_layout()

## 5. Several motions at once

Give `run` a list and it simulates the motions in parallel worker processes (one per motion, up to your CPU count), with one progress bar per motion. Each result is identical to running that motion alone.

Here: the same minute of all three DB5 exercises.

In [ ]:
files = sorted(DB5.glob("*.mat"))
minutes = [segment(mg.HandMotion.load(f, load_db5), window) for f in files]
recordings = sim.run(minutes)
for f, r in zip(files, recordings):
    print(f"{f.stem}: {r.duration:.0f} s, QC pass {r.qc['pass']}, tracking RMSE {r.qc['rmse_deg']:.2f} deg")

## 6. Real vs simulated EMG

Both are Myo int8 counts at 200 Hz, so they can share axes. Raw EMG is noise-like: two recordings of the same movement never match sample by sample. What should match is the **envelope** (the moving RMS, i.e. how strongly each channel is active over time).

In [ ]:
real = rec["emg"][window].astype(float)
simulated = recording.emg.astype(float)
n = min(len(real), len(simulated))
real, simulated, t_emg = real[:n], simulated[:n], np.arange(n) / fs


def envelope(x, fs=200.0, window_s=0.15):
    """Moving RMS per channel."""
    w = np.ones(int(window_s * fs)) / int(window_s * fs)
    return np.sqrt(np.apply_along_axis(lambda c: np.convolve(c ** 2, w, "same"), 0, x))


def compare_plot(real, simulated, channels=(0, 3, 9, 12), title=""):
    env_r, env_s = envelope(real), envelope(simulated)
    fig, ax = plt.subplots(len(channels) + 1, 2, figsize=(12, 2 * len(channels) + 2), sharex=True)
    for row, c in enumerate(channels):
        ax[row, 0].plot(t_emg, real[:, c], lw=0.3, color="0.3", label="real")
        ax[row, 0].plot(t_emg, simulated[:, c], lw=0.3, color="C1", alpha=0.8, label="simulated")
        ax[row, 0].set(ylabel=f"ch{c}", ylim=(-128, 128))
        ax[row, 1].plot(t_emg, env_r[:, c], color="0.3", lw=1)
        ax[row, 1].plot(t_emg, env_s[:, c], color="C1", lw=1)
        r = np.corrcoef(env_r[:, c], env_s[:, c])[0, 1]
        ax[row, 1].text(0.01, 0.85, f"r = {r:.2f}", transform=ax[row, 1].transAxes, fontsize=8)
    ax[-1, 0].plot(t_emg, rec["restimulus"][window][:n], color="k", lw=1)
    ax[-1, 0].set(ylabel="movement", xlabel="time (s)")
    ax[-1, 1].plot(t_emg, env_r.mean(1), color="0.3", lw=1.2, label="real")
    ax[-1, 1].plot(t_emg, env_s.mean(1), color="C1", lw=1.2, label="simulated")
    r = np.corrcoef(env_r.mean(1), env_s.mean(1))[0, 1]
    ax[-1, 1].set(ylabel="mean of 16 ch", xlabel="time (s)", title=f"mean envelope, r = {r:.2f}")
    ax[0, 0].set_title("raw EMG (counts)")
    ax[0, 1].set_title("envelope (150 ms RMS)")
    ax[0, 0].legend(loc="upper right", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()


compare_plot(real, simulated, title="real vs simulated, default device settings")

In [ ]:
# The same comparison as numbers (mg.compare: signal statistics of each, plus the envelope correlation per channel).
for k, (r, s) in mg.compare(real, simulated, fs, "myo", rest=rec["restimulus"][window][:n] == 0).items():
    print(f"{k:22s} real {'-' if r is None else f'{r:.3g}':>8}   simulated {s:.3g}")

## 7. Calibrating the device to the real recording

The default device settings (gain, noise floor) and the armband's rotation around the forearm are guesses. `mg.calibrate` fits them to a real recording of the same motion: it simulates the muscle activity once, then tries armband rotations (in parallel) and fits gain and noise, re-running only the cheap device step.

To use the result without simulating again: `simulate_clean` gives the device-independent part (muscle activity + clean EMG) and `record` puts it through any device settings.

In [ ]:
cal = mg.calibrate(sim, [minute], [rec["emg"][window]], fit_rotation=True)
print(cal)

In [ ]:
calibrated = mg.Simulator(cal.electrodes, subject=subject)
recording_cal = calibrated.run(minute)
compare_plot(real, recording_cal.emg[:n].astype(float), title="real vs simulated, calibrated device")

After calibration the amplitudes and the noise floor match the real recording, and the envelopes rise and fall with the same movements on most channels. Channel by channel they still differ: the glove-to-joint retargeting and the muscle model are approximations, so read the envelope correlations as a measure of how close the simulation gets, not as an error to drive to zero. (The burst in the first half second is the simulation settling into the first posture.)

## 8. Saving, loading, and the 3-D replay

Recordings are HDF5 files with the EMG, the labels, the electrodes and the subject. `mg.replay` opens one in a web viewer: the hand moving, the muscles lighting up with their activity, the electrodes and their EMG.

The replay keeps running: open http://localhost:8080 in your browser, and **interrupt the kernel** (■) to stop it.

In [ ]:
path = recording_cal.save("outputs/s1_e1_minute.h5")
again = mg.Recording.load(path)
print(path, f"{again.duration:.0f} s,", len(again.electrodes), "channels")

In [ ]:
mg.replay(again)          # http://localhost:8080 — interrupt the kernel to stop